# Tree sequences with RELATE

In [ ]:
# load in the relevant packages
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd 
import numpy as np
from pathlib import Path
from geneinfo.plot import gene_plot
%config InlineBackend.figure_format = 'svg'
from vscodenb import set_vscode_theme, vscode_theme
set_vscode_theme()
sns.set_palette('tab10')
sns.palplot(sns.color_palette(), size=0.2)

And a bit to make the plots look nicer in vscode:

If the sequences in your data set are individual haplotypes (they are phased), it is possible to construct the coalescence trees for each non-recombining segment of a genomic alignment. If you are working on male X chromosomes this is easy because they are haploid and do not require phasing. However, autosomes are diploid, and we would like to use haplotype-based analysis here as well. This can be done through either phasing of short reads mapped to a reference genome, or by assembing each haplotype `de novo` using long reads such as PacBio HiFi. Your chr2 data set is already phased so you are good to go.

In this exercise we will run the Relate program to infer local trees. In the upcomming exercises we will revisit this tree sequence and use it for inference of demography and selection. 

## Request an interactive session on a compute node:

Begin by requesting a shell on a compute node:


## Data files

The chr2 data for this exerise is from 60 individuals in the 1000Genomes project. There are 20 individuals from each of the following populations: GBR (British in England and Scotland), JPT (Japanese), YRI (Yoruban).

Create some symlinks that point to the following files:



This way it looks like the files are in your current folder. You can run `ls` to see them. The first file is a mask of genomic regions that either have abnormal read depth or contain repetitive elements. The second file is a recombination map. The third file is the ancestral state of every site, based on an alignment with gorilla, chimpanzee and human genomes. The fourth file is a metadata file detailing the population and region for each sample. The last file is the phased genotype VCF file.

The documentation for Relate can be found [here](https://myersgroup.github.io/relate/).

The `chr2_130_145_phased.vcf.gz` has individuals from three populations. To make tree sequences for each population, we first need to split the file into three files with samples from each population:


> **NB:** Running Relate below, you should be aware that names of input files are always supplied without the file extensions.

Relate does not accept the standard VCF file format, but instead uses a haps/sample format. You can read up on in the Relate documentation. The authors have been so kind as to supply a script to transform it. First, the vcf is converted to another file format (haplotype file format). If you want to know how it is structured, you can read about it [here](https://www.cog-genomics.org/plink/2.0/formats#haps).

Then, repetitive (unreliably sequenced) regions must be masked to exclude them form our analysis. We also need to assign each variant as either ancestral or derived using the chimpanzee genome. We do both with this command:



Inspect the generated files. 


**How many SNPs were removed in this filtering step?**

Hint: `cat chr2.haps | wc -l` counts the number of lines in `chr2.haps`. And `zcat` is `cat` for `.gz` files.


## Run RELATE to build trees along the genome

Now, the input is fully prepared, and Relate can be run. This should take less than a minute.

**How many SNPs are left per haplotype?**


## Estimate historical population size and re-estimate branch lengths of trees

The lengthiest process is this step, in which population size is estimated, and the population size is re-estimates branch lengths. This takes around 20 minutes. While you are waiting, explain to a fellow student how an ARG can be constructed backwards in time and how it can be constructed along the sequence. If time permits, make sure to also explain how the SMC and SMC' models approximate the ARG.

Relate outputs estimated mutation rate and coalescence times along the region.


# Positive selection

In this exercise we will work on inference of selection using the Relate tool that we also used in the [exercise about tree sequences](https://github.com/kaspermunch/PopulationGenomicsCourse/tree/master/Exercises/tree_sequences). In that exercise we analyzed individuals from three populations and inferred all the trees along the chr2 region we have been working on. In this exercise we are going to use those trees to take the analysis a step further and look for positive selection in our chr2 region. So we pick up the [tree sequences exercise](https://github.com/kaspermunch/PopulationGenomicsCourse/tree/master/Exercises/tree_sequences) where we left it and you should continue this exericse in the same folder. That way you have access to the files you you produced the other exercise.

Since we already have the trees along our genomic alignment we can now use Relate to analyze each tree and compute the likelihood that each mutation on it was shaped by positive selection. Make sure you understand how Relate quantifies evidence of selection and how that lines up with what you know about positive selection. Team up with one or more fellow students and make sure you understand the relevant Methods section in the [Relate paper](https://www.nature.com/articles/s41588-019-0484-x) as well as the section "A tree-based statistic for detecting positive selection" in the [supplementary note for the paper](https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-019-0484-x/MediaObjects/41588_2019_484_MOESM1_ESM.pdf).

The Relate command runs the test for positive selection on each SNP in each population:

Have a look at the `gbr_relate.sele` produced, and see what is in there.

In [ ]:
gbr_df = pd.read_csv('gbr_relate.sele', sep=' ')
gbr_df.head()

You can use the `usecols` argument to specify which columns to read in, which can speed up the loading process if the file is large and you only need a subset of the columns. Read in the files for both the GBR, JPT and YRI populations:

In [ ]:
cols = ['pos', 'rs_id', 'when_mutation_has_freq2']
gbr_df = pd.read_csv('gbr_relate.sele', sep=' ', usecols=cols).set_index('rs_id')
jpt_df = pd.read_csv('jpt_relate.sele', sep=' ', usecols=cols).set_index('rs_id')
yri_df = pd.read_csv('yri_relate.sele', sep=' ', usecols=cols).set_index('rs_id')

gbr_df.head()

The last column is the log10 p-value for the test for selection described in the paper.

In [ ]:


with vscode_theme(style='ticks'):
    ax = gene_plot('chr2', 130_000_000, 145_000_000, 'hg38', figsize=(8, 8))
    ax.scatter(gbr_df.pos, - gbr_df.when_mutation_has_freq2, s=2, c='tab:blue', label='GBR')
    ax.scatter(jpt_df.pos, - jpt_df.when_mutation_has_freq2, s=2, c='tab:orange', label='JPT')
    ax.scatter(yri_df.pos, - yri_df.when_mutation_has_freq2, s=2, c='tab:green', label='YRI')
    ax.set_ylim(bottom=0)
    ax.legend() ;

In [ ]:
with vscode_theme(style='ticks'):
    ax = gene_plot('chr2', 135_000_000, 136_500_000, 'hg38', figsize=(8, 8))
    ax.scatter(gbr_df.pos, - gbr_df.when_mutation_has_freq2, s=2, c='tab:blue', label='GBR')
    ax.scatter(jpt_df.pos, - jpt_df.when_mutation_has_freq2, s=2, c='tab:orange', label='JPT')
    ax.scatter(yri_df.pos, - yri_df.when_mutation_has_freq2, s=2, c='tab:green', label='YRI')
    ax.set_ylim(bottom=0)
    ax.legend() ;

In [ ]:
with vscode_theme(style='ticks'):
    ax = gene_plot('chr2', 135_000_000, 136_500_000, 'hg38', figsize=(8, 8), return_axes=3, aspect=0.5)

    ax[0].scatter(gbr_df.pos, - gbr_df.when_mutation_has_freq2, s=2, c='tab:blue', label='GBR')
    ax[0].set_ylim(0, 6)

    ax[1].scatter(jpt_df.pos, - jpt_df.when_mutation_has_freq2, s=2, c='tab:orange', label='JPT')
    ax[1].set_ylim(0, 6)

    ax[2].scatter(yri_df.pos, - yri_df.when_mutation_has_freq2, s=2, c='tab:green', label='YRI')
    ax[2].set_ylim(0, 6)



## Analyzing tree sequences

`tskit` is a powerful library for working with tree sequences, which are data structures that efficiently represent the genealogical history of a set of DNA sequences. To load a tree sequence from a file, you can use the `tskit.load()` function. You can read all about `tskit` in its [documentation](https://tskit.dev).

Load the tree sequence for the GBR population:

In [ ]:
import tskit
ts = tskit.load(f'gbr_relate.trees')

### Mutations and Sites

Mutations live on sites, and sites have positions. A site can have multiple mutations. Get the site at at some position (in this case, you get an error because there is no site at the position):

In [ ]:
try:
    site = ts.site(position=130000000)  # looks up by position
except ValueError:
    print("No site at that position")

If you do not have the position of a site, you can search by approximate position or find the nearest site (SNP):

In [ ]:
pos = 130000000
positions = ts.sites_position  # array of all site positions
idx = np.searchsorted(positions, pos) # get the index of the site closest to that position
site = ts.site(idx) # get the site object using the index
site

Now you can get the mutations at that site:

In [ ]:
for mut in site.mutations:
    print(site.position, mut.id, mut.node, mut.derived_state)

There is almost always just one mutation at a site in which case you can just do:

In [ ]:
mut = site.mutations[0]
print(site.position, mut.id, mut.node, mut.derived_state)

### Trees

Find the position of the SNP with the smallest p-value:

In [ ]:
idx = np.argmin(gbr_df.when_mutation_has_freq2)
pos = gbr_df.iloc[idx].pos.item()
pos

Then get the tree that covers the position:

In [ ]:
tree = ts.at(pos)
tree

How many samples carry this mutation with the lowest p-value?

In [ ]:
list(tree.get_leaves(mut.node))

What is the age of the mutation at this position? 

In [ ]:
tree.time(mut.node)

### Plotting trees

The code below defines the `plot_tree` that you can use to visualize trees:

In [ ]:
from itertools import takewhile
import matplotlib
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from collections import defaultdict

def in_dark_theme(ax=None):
    """Returns True if the background is dark, False otherwise."""
    if ax is None:
        plt.ioff()
        ax = plt.gca()
        plt.ion()        
    bg_color = ax.get_facecolor()
    # Convert to grayscale to determine brightness
    luminance = matplotlib.colors.rgb_to_hsv(matplotlib.colors.to_rgb(bg_color))[2]
    return bool(luminance < 0.5)


def plot_tree(t, 
              ax=None, 
              leaf_colors=None, 
              show_node_ids=False, 
              show_mutations=True,
              fontsize=7, 
              alpha=1,
              leaf_order=None,
              margins=(0.5, 1, 0.5, 1), # top, right, bottom, left
              invert=False,
              log_scale=False,
              return_leaf_coords=False,
              align_labels=False,
              **kwargs): 

    sign = 2 * int(invert) - 1

    if leaf_order is None:
        leaf_order = []
        for node in t.nodes(order="postorder"):        
            if t.is_leaf(node):
                leaf_order.append(node)

    return_ax = False
    if ax is None:   
        if 'figsize' not in kwargs:
            kwargs['figsize'] = (8, 5)
        return_ax = True
        fig, ax = plt.subplots(**kwargs)

    if log_scale:
        ax.set_xscale('symlog', base=10)

    branch_muts = defaultdict(list)
    for site in t.sites():
        for mut in site.mutations:
            branch_muts[mut.node].append(mut)

    x_offsets = {}
    y_offsets = {}

    y_offset = len(list(t.leaves(t.root)))
    for node in t.nodes(order="preorder"):
        x_offsets[node] = t.time(node) 

    for node in leaf_order:
        y_offset -= 1
        y_offsets[node] = y_offset
            
    for node in t.nodes(order="postorder"):        
        if not t.is_leaf(node):
            children = t.children(node)
            y_offsets[node] = sum(y_offsets[x] for x in children) / len(children)

    horizontal_lines = list()
    vertical_lines = list()
    node_coords = list()
    leaf_coords = list()
    node_labels = list()
    max_x_offset = 0
    mut_positions = defaultdict(list)
    for node in t.nodes(order="postorder"):        
        max_x_offset = max(max_x_offset, x_offsets[node])
        node_coords.append((x_offsets[node], y_offsets[node]))
        if t.is_leaf(node):
            leaf_coords.append([str(node), x_offsets[node], y_offsets[node]])
        if not t.is_root(node):
            y = y_offsets[node]
            horizontal_lines.append(([x_offsets[t.parent(node)], x_offsets[node]], [y, y]))

            muts = branch_muts[node]
            parent = t.parent(node)
            t_child = t.time(node)
            t_parent = t.time(parent)
            n = len(muts)
            for i, mut in enumerate(muts):
                frac = (i + 1) / (n + 1)
                x = t_child + frac * (t_parent - t_child)
                # mut_positions[mut.id] = (-x, y)        
                mut_positions[node].append((x, y))    

        if not t.is_leaf(node):
            c = sorted(t.children(node), key=lambda x: y_offsets[x])
            bottom, top = c[0], c[-1]
            x = x_offsets[node]
            vertical_lines.append(([x, x],[y_offsets[bottom], y_offsets[top]]))
            y = (y_offsets[bottom] + y_offsets[top]) / 2

            # strings =[str(x) for x in c]
            # prefix = ''.join(c[0] for c in takewhile(lambda x: all(x[0] == y for y in x), zip(*strings)))

            # df = data.loc[data.haplotype.str.startswith(prefix)]
            # suffix = f'({df.loc[df.case == 1].index.size}/{df.loc[df.case == 0].index.size})'

            # label  = prefix + ' ' + suffix
            # node.name = prefix
            # node_labels.append((-x, y, str(node)))
            node_labels.append((x, y, str(node)))

    if in_dark_theme(ax):
        foreground_color = 'white'
        label_color = 'grey'
        mut_color = 'hotpink'
    else:
        foreground_color = 'black'
        label_color = 'lightgrey'
        mut_color = 'hotpink'

    # draw the tree:
    for x in horizontal_lines:
        ax.plot(*x, c=foreground_color, linewidth=0.8, alpha=alpha)
    for x in vertical_lines:
        ax.plot(*x, c=foreground_color, linewidth=0.8, alpha=alpha)

    if show_node_ids:
        d = 0#max_x_offset / 300
        for x, y, txt in node_labels:
            ax.text((x-d), y, txt, fontsize=fontsize,
                    horizontalalignment='left' if invert else 'right',
                    verticalalignment='bottom', color=label_color)

    for name, x, y in leaf_coords:
                
        if align_labels:
            ax.text(0, y, name, fontsize=fontsize,
                    verticalalignment='center', horizontalalignment='right' if invert else 'left')
            if leaf_colors is None:
                color = 'black'
            else:
                color = leaf_colors[name]
            # ax.plot(x, y, c=color, marker="o", ms=3)
            ax.add_line(Line2D((x*sign, sign), (y, y), linewidth=0.8, color='grey', linestyle='dashed', zorder=0))
        else:
            ax.text(x, y, name, fontsize=fontsize,
                    verticalalignment='center', horizontalalignment='right' if invert else 'left'
                    )
            if leaf_colors is None:
                color = 'black'
            else:
                color = leaf_colors[name]
            # ax.plot(x, y, c=color, marker="o", ms=3)

    if show_mutations:
        mutation_markers = []
        for node in t.nodes(order="postorder"):
            if node in mut_positions:
                mutation_markers.extend(mut_positions[node])
        if mutation_markers:
            ax.scatter(*zip(*mutation_markers), c=mut_color, marker="o", s=10, zorder=10)

    # ax.set_xlim(-margins[3]-max_x_offset, margins[1])
    # ax.set_ylim(-margins[2], len(leaf_coords)-1+margins[0])

    # ax.set_xlim(margins[3]+max_x_offset, margins[1])
    ax.set_xlim(abs(max_x_offset), 0)
    ax.set_ylim(-margins[2], len(leaf_coords)-1+margins[0])

    ax.get_yaxis().set_visible(False)

    ax.spines['top'].set_visible(False) 
    ax.spines['left'].set_visible(False) 
    ax.spines['right'].set_visible(False)
    
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
    if invert:
        ax.invert_xaxis()

    ret_val = []
    if return_ax:
        ret_val.append(ax)
    if return_leaf_coords:
        ret_val.append(leaf_coords)
    if ret_val:
        return ret_val
    # plt.tight_layout()
    # plt.show()    

In [ ]:
plot_tree(tree) ;

In [ ]:
plot_tree(tree, figsize=(10, 4), show_node_ids=True, log_scale=True) ;

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 4))
plot_tree(tree, ax=ax1, log_scale=True)
plot_tree(tree, ax=ax2, invert=True, log_scale=True)
plt.tight_layout()

In [ ]:
tree# = ts.at(pos)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5, 5))

plot_tree(tree, ax=ax1, log_scale=True)
plot_tree(ts.at(tree.index+1), leaf_order=list(tree.leaves(tree.root)), ax=ax2, log_scale=True,invert=True)
plt.tight_layout()

## Full chr2 analysis of full 1000 genomes data set

| Tag | Name                 | Region    | Location                      |
|:----|:---------------------|:----------|:------------------------------|
| GWD | Gambian              | Africa    | Gambian in Western Division, The Gambia  |
| MSL | Mende                | Africa    | Mende in Sierra Leone |
| ESN | Esan                 | Africa    | Esan in Nigeria |
| LWK | Luhya                | Africa    | Luhya in Webuye, Kenya |
| YRI | Yoruba               | Africa    | Yoruba in Ibadan, Nigeria |
| CEU | CEPH                 | Europe    | Utah residents (CEPH) with Northern and Western European ancestry  |
| TSI | Tuscan               | Europe    | Toscani in Italia  |
| GBR | British              | Europe    | British in England and Scotland  |
| FIN | Finnish              | Europe    | Finnish in Finland  |
| IBS | Spanish              | Europe    | Iberian populations in Spain  |
| GIH | Gujarati             | SouthAsia | Gujarati Indian in Houston, TX |
| PJL | Punjabi              | SouthAsia | Punjabi in Lahore, Pakistan |
| BEB | Bengali              | SouthAsia | Bengali in Bangladesh |
| STU | Sri Lankan           | SouthAsia | Sri Lankan Tamil in the UK |
| ITU | Indian               | SouthAsia | Indian Telugu in the UK |
| CHB | Han Chinese          | EastAsia  | Han Chinese in Beijing, China |
| JPT | Japanese             | EastAsia  | Japanese in Tokyo, Japan |
| CHS | Southern Han Chinese | EastAsia  | Han Chinese South |
| CDX | Dai Chinese          | EastAsia  | Chinese Dai in Xishuangbanna, China |
| KHV | Kinh Vietnamese      | EastAsia  | Kinh in Ho Chi Minh City, Vietnam |
| CHD | Denver Chinese       | EastAsia  | Chinese in Denver, Colorado (pilot 3 only) |
| ASW | African-American SW  | America   | African Ancestry in Southwest US   |
| ACB | African-Caribbean    | America   | African Caribbean in Barbados |
| MXL | Mexican-American     | America   | Mexican Ancestry in Los Angeles, California |
| PUR | Puerto Rican         | America   | Puerto Rican in Puerto Rico |
| CLM | Colombian            | America   | Colombian in Medellin, Colombia |
| PEL | Peruvian             | America   | Peruvian in Lima, Peru |

In [ ]:
chr2_1000g = pd.read_parquet('relate_log10pvals_1000g_pops.parquet')

In [ ]:
snp_pos = 0
with vscode_theme(style='ticks'):
    pops = ['GBR', 'FIN', 'JPT', 'GWD', 'MSL', 'ESN', 'LWK','YRI']
    fig, axes = plt.subplots(4, 2, figsize=(7, 7), sharex=True, sharey=True)
    axes[0, 0].set_xlim(132_000_000, 138_000_000)
    for pop, ax in zip(pops, axes.flat):
        ax.scatter(chr2_1000g.pos, - chr2_1000g[pop], s=2)
        ax.set_title(pop)
        ax.axhline(y=8.5, color='tab:orange', linestyle='dotted', lw=1, alpha=0.5)
        ax.axvline(x=snp_pos, color='red', linestyle='dashed')

    plt.tight_layout()

In [ ]:
with vscode_theme(style='ticks'):
    ax = gene_plot('chr2', 132_000_000, 138_500_000, 'hg38', figsize=(8, 6))
    ax.scatter(chr2_1000g.pos, - chr2_1000g.GBR, s=2, c='tab:blue', label='GBR')
    ax.scatter(chr2_1000g.pos, - chr2_1000g.JPT, s=2, c='tab:orange', label='JPT')
    ax.scatter(chr2_1000g.pos, - chr2_1000g.YRI, s=2, c='tab:green', label='YRI')
    ax.set_ylim(bottom=0)
    ax.legend(loc='upper right') ;